In [3]:
"""
PYNQ-Z2 Script: PS → AXI SmartConnect → AXI BRAM Controller → BRAM
                → BRAM Reader → AXI DMA → PS

BRAM Reader parameters (from Verilog):
    DATA_WIDTH = 16 bits
    M          = 9  (parallel lanes)
    count      = 16 (number of addresses to read)
    Word width = DATA_WIDTH * M = 144 bits = 18 bytes per transfer
    Total DMA  = 16 words × 18 bytes = 288 bytes
"""

from pynq import Overlay, allocate
import numpy as np
import time

# ──────────────────────────────────────────────
# 1.  Load the bitstream
# ──────────────────────────────────────────────
BITSTREAM = "design_2_wrapper.bit"          # <-- change to your actual .bit file name
ol = Overlay(BITSTREAM)

print("Overlay loaded successfully.")
print(ol.ip_dict.keys())            # lists all IPs in the overlay

# ──────────────────────────────────────────────
# 2.  Grab IP handles
# ──────────────────────────────────────────────
# AXI BRAM Controller  (mapped to BRAM Port A)
bram_ctrl = ol.axi_bram_ctrl_0      # adjust name to match your HWH

# AXI DMA  (receives the stream from bram_reader via S2MM channel)
dma = ol.axi_dma_0                  # adjust name to match your HWH

# ──────────────────────────────────────────────
# 3.  BRAM layout constants
# ──────────────────────────────────────────────
DATA_WIDTH   = 16          # bits per lane
M            = 9           # parallel lanes
COUNT        = 16          # addresses the BRAM reader will iterate through
WORD_BYTES   = (DATA_WIDTH * M) // 8   # 18 bytes  (144 bits / 8)
TOTAL_BYTES  = COUNT * WORD_BYTES      # 288 bytes total DMA transfer

# The AXI BRAM Controller exposes a flat byte-addressable window.
# Each BRAM address holds one 144-bit word = 18 bytes.
# We write COUNT words starting at offset 0.

# ──────────────────────────────────────────────
# 4.  Write test data into BRAM (PS → BRAM Controller → BRAM Port A)
# ──────────────────────────────────────────────
print("\nWriting test data to BRAM ...")

# Build COUNT words.  Each word = M=9 lanes × 16 bits.
# We store them as uint16 arrays then write to BRAM byte-by-byte (32-bit AXI).
rng = np.random.default_rng(42)
input_data = rng.integers(0, 0xFFFF, size=(COUNT, M), dtype=np.uint16)

for addr_idx in range(COUNT):
    byte_offset = addr_idx * WORD_BYTES   # byte address in BRAM window
    word = input_data[addr_idx]           # shape (M,) uint16

    # AXI BRAM Controller is 32-bit wide on the AXI side.
    # Pack the 18-byte word into 5 × 32-bit writes (last write uses only 2 bytes).
    raw_bytes = word.tobytes()            # 18 bytes, little-endian

    # Pad to a multiple of 4 bytes so we can cast to uint32
    padded = raw_bytes + b'\x00' * (4 - len(raw_bytes) % 4)   # pad to 20 bytes
    words32 = np.frombuffer(padded, dtype=np.uint32)           # 5 × 32-bit words

    for i, w32 in enumerate(words32):
        bram_ctrl.write(byte_offset + i * 4, int(w32))

print(f"  Written {COUNT} words ({WORD_BYTES} bytes each) to BRAM.")

# ──────────────────────────────────────────────
# 5.  Allocate a DMA receive buffer (S2MM: stream → memory)
# ──────────────────────────────────────────────
recv_buf = allocate(shape=(TOTAL_BYTES,), dtype=np.uint8)
recv_buf[:] = 0
print(f"\nDMA receive buffer allocated: {TOTAL_BYTES} bytes")

# ──────────────────────────────────────────────
# 6.  Start the DMA S2MM transfer and wait
#     The BRAM reader starts automatically after reset de-assertion;
#     it streams COUNT words then asserts tlast.
# ──────────────────────────────────────────────
print("Starting DMA S2MM transfer ...")
dma.recvchannel.transfer(recv_buf)
dma.recvchannel.wait()          # blocks until tlast is received
print("DMA transfer complete.")

# ──────────────────────────────────────────────
# 7.  Decode and verify the received data
# ──────────────────────────────────────────────
print("\n── Received data (first 4 words) ──")
received = np.frombuffer(recv_buf, dtype=np.uint8)

for addr_idx in range(COUNT):
    start_byte = addr_idx * WORD_BYTES
    raw = received[start_byte : start_byte + WORD_BYTES]   # 18 bytes
    # Reinterpret as uint16 (little-endian); last 2 bytes = lane 8
    word_rx = np.frombuffer(raw.tobytes(), dtype=np.uint16)  # shape (9,)

    if addr_idx < 4:
        print(f"  Word[{addr_idx:02d}] TX: {input_data[addr_idx]}")
        print(f"           RX: {word_rx}")
        match = np.array_equal(input_data[addr_idx], word_rx)
        print(f"           Match: {'✓' if match else '✗ MISMATCH'}")

# Full match check
all_ok = True
for addr_idx in range(COUNT):
    start_byte = addr_idx * WORD_BYTES
    raw = received[start_byte : start_byte + WORD_BYTES]
    word_rx = np.frombuffer(raw.tobytes(), dtype=np.uint16)
    if not np.array_equal(input_data[addr_idx], word_rx):
        print(f"[FAIL] Mismatch at word {addr_idx}")
        all_ok = False

if all_ok:
    print(f"\n✓ All {COUNT} words match — BRAM write → BRAM reader → DMA pipeline OK.")
else:
    print("\n✗ Some words did not match. Check BRAM addressing / DMA byte order.")

# ──────────────────────────────────────────────
# 8.  Clean up
# ──────────────────────────────────────────────
recv_buf.freebuffer()
print("\nDone.")

RuntimeError: Unable to find metadata for bitstream

In [4]:
import os

print("Current directory:", os.getcwd())
print("\nAll files here:")
for f in sorted(os.listdir('.')):
    print(f"  {f}")

Current directory: /home/xilinx/jupyter_notebooks/tpu

All files here:
  .ipynb_checkpoints
  Untitled.ipynb
  design_2_wrapper.bit


In [5]:
import subprocess

result = subprocess.run(
    ['find', '/', '-name', '*.hwh', '-type', 'f'],
    capture_output=True, text=True, timeout=10
)
print("Found .hwh files:")
print(result.stdout if result.stdout else "NONE FOUND")

TimeoutExpired: Command '['find', '/', '-name', '*.hwh', '-type', 'f']' timed out after 10 seconds

In [5]:
from pynq import Overlay, allocate, MMIO
import numpy as np
import time

# ──────────────────────────────────────────────
# 1. Load Bitstream (ignore metadata issues)
# ──────────────────────────────────────────────
ol = Overlay("design_2_wrapper.bit", download=True)

print("Overlay loaded (metadata bypassed).")

# DMA still works because it is exposed
dma = ol.axi_dma_0

# ──────────────────────────────────────────────
# 2. Manually define BRAM base address
# ──────────────────────────────────────────────
BRAM_BASE  = 0x40000000   # from your Address Editor
BRAM_RANGE = 0x10000

bram = MMIO(BRAM_BASE, BRAM_RANGE)

# ──────────────────────────────────────────────
# 3. Constants (match RTL)
# ──────────────────────────────────────────────
DATA_WIDTH  = 16
M           = 9
COUNT       = 16

WORD_BYTES  = (DATA_WIDTH * M) // 8        # 18 bytes  (raw word size)
WORD_STRIDE = WORD_BYTES + (-WORD_BYTES % 4)  # 20 bytes  (aligned stride for BRAM writes)
TOTAL_BYTES = COUNT * WORD_BYTES           # 288 bytes (DMA transfer, no padding)

# ──────────────────────────────────────────────
# 4. Write data to BRAM
# ──────────────────────────────────────────────
print("\nWriting data to BRAM...")

rng = np.random.default_rng(42)
input_data = rng.integers(0, 0xFFFF, size=(COUNT, M), dtype=np.uint16)

for addr_idx in range(COUNT):
    byte_offset = addr_idx * WORD_STRIDE   # ← 0, 20, 40, 60... all ÷4 ✓

    raw_bytes = input_data[addr_idx].tobytes()          # 18 bytes
    pad_len   = (-len(raw_bytes)) % 4                   # 2 bytes
    padded    = raw_bytes + b'\x00' * pad_len           # 20 bytes
    words32   = np.frombuffer(padded, dtype=np.uint32)  # 5 × 32-bit

    for i, w32 in enumerate(words32):
        bram.write(byte_offset + i * 4, int(w32))       # always aligned ✓

print("BRAM write complete.")

# ──────────────────────────────────────────────
# 5. Allocate DMA buffer
# ──────────────────────────────────────────────
recv_buf = allocate(shape=(TOTAL_BYTES,), dtype=np.uint8)
recv_buf[:] = 0

# ──────────────────────────────────────────────
# 6. Start DMA
# ──────────────────────────────────────────────
print("\nStarting DMA transfer...")

dma.recvchannel.stop()
dma.recvchannel.start()

dma.recvchannel.transfer(recv_buf)

# timeout protection
timeout = 5
start = time.time()

while not dma.recvchannel.idle:
    if time.time() - start > timeout:
        raise RuntimeError("DMA stuck! Check tvalid/tlast")

print("DMA complete.")

# ──────────────────────────────────────────────
# 7. Verify data
# ──────────────────────────────────────────────
print("\nVerifying data...\n")

received = np.frombuffer(recv_buf, dtype=np.uint8)

all_ok = True

for addr_idx in range(COUNT):
    start_byte = addr_idx * WORD_BYTES
    raw = received[start_byte:start_byte + WORD_BYTES]

    word_rx = np.frombuffer(raw.tobytes(), dtype=np.uint16)

    if addr_idx < 4:
        print(f"Word[{addr_idx}]")
        print("TX:", input_data[addr_idx])
        print("RX:", word_rx)

    if not np.array_equal(input_data[addr_idx], word_rx):
        print(f"Mismatch at word {addr_idx}")
        all_ok = False

if all_ok:
    print("\nSUCCESS: Data matches!")
else:
    print("\nERROR: Data mismatch!")

# ──────────────────────────────────────────────
# 8. Cleanup
# ──────────────────────────────────────────────
recv_buf.freebuffer()

print("\nDone.")

Overlay loaded (metadata bypassed).

Writing data to BRAM...
BRAM write complete.

Starting DMA transfer...


RuntimeError: DMA stuck! Check tvalid/tlast

In [6]:
from pynq import Overlay, allocate, MMIO
import numpy as np
import time

# ──────────────────────────────────────────────
# 1. Load Bitstream
# ──────────────────────────────────────────────
ol = Overlay("design_2_wrapper.bit", download=True)
print("Overlay loaded.")

# ──────────────────────────────────────────────
# 2. IP Handles via MMIO
# ──────────────────────────────────────────────
# DMA
dma = ol.axi_dma_0

# BRAM Controller (check your Address Editor)
BRAM_BASE  = 0x40000000
BRAM_RANGE = 0x10000
bram = MMIO(BRAM_BASE, BRAM_RANGE)

# DMA base address (check your Address Editor)
DMA_BASE = 0x40400000
dma_mmio = MMIO(DMA_BASE, 0x1000)

# ──────────────────────────────────────────────
# 3. Constants
# ──────────────────────────────────────────────
DATA_WIDTH  = 16
M           = 9
COUNT       = 16

WORD_BYTES  = (DATA_WIDTH * M) // 8           # 18 bytes raw
WORD_STRIDE = WORD_BYTES + (-WORD_BYTES % 4)  # 20 bytes aligned
TOTAL_BYTES = COUNT * WORD_BYTES              # 288 bytes for DMA

# ──────────────────────────────────────────────
# 4. Generate and write data to BRAM
# ──────────────────────────────────────────────
print("\nWriting data to BRAM...")

rng = np.random.default_rng(42)
input_data = rng.integers(0, 0xFFFF, size=(COUNT, M), dtype=np.uint16)

for addr_idx in range(COUNT):
    byte_offset = addr_idx * WORD_STRIDE      # 0, 20, 40 ... always ÷4
    raw_bytes   = input_data[addr_idx].tobytes()
    pad_len     = (-len(raw_bytes)) % 4
    padded      = raw_bytes + b'\x00' * pad_len
    words32     = np.frombuffer(padded, dtype=np.uint32)

    for i, w32 in enumerate(words32):
        bram.write(byte_offset + i * 4, int(w32))

print("✓ BRAM write complete.")

# ──────────────────────────────────────────────
# 5. Allocate DMA receive buffer
# ──────────────────────────────────────────────
recv_buf = allocate(shape=(TOTAL_BYTES,), dtype=np.uint8)
recv_buf[:] = 0
print(f"✓ DMA buffer allocated: {TOTAL_BYTES} bytes")

# ──────────────────────────────────────────────
# 6. Reset DMA S2MM channel cleanly
# ──────────────────────────────────────────────
# S2MM Control register offset = 0x30
# Bit 2 = reset, Bit 0 = run/stop
S2MM_CTRL = 0x30
S2MM_STAT = 0x34

# Issue soft reset
dma_mmio.write(S2MM_CTRL, 0x4)
time.sleep(0.1)

# Wait for reset to clear
for _ in range(100):
    if not (dma_mmio.read(S2MM_CTRL) & 0x4):
        break
    time.sleep(0.01)

# Start S2MM (Run bit=1)
dma_mmio.write(S2MM_CTRL, 0x1)
time.sleep(0.05)
print("✓ DMA S2MM channel started.")

# ──────────────────────────────────────────────
# 7. ARM the DMA BEFORE the BRAM reader streams
#    Critical order: arm DMA → then trigger BRAM reader
# ──────────────────────────────────────────────
print("\nArming DMA transfer...")
dma.recvchannel.transfer(recv_buf)   # DMA now waiting for tvalid

# ──────────────────────────────────────────────
# 8. Trigger BRAM reader by toggling its reset
#    via the GPIO (if connected) or reload overlay
#    Here we reload just the DMA to re-trigger start
# ──────────────────────────────────────────────
# The BRAM reader's `start` goes high on rst=1 then streams on rst=0.
# Since rst is tied to peripheral_aresetn (active-low = 0 means reset),
# the reader starts automatically after bitstream load.
# If DMA was armed fast enough it should catch the stream.
# If not, we need a GPIO. Check if one exists:

print("Checking for GPIO reset control...")
ip_names = list(ol.ip_dict.keys())
print("Available IPs:", ip_names)

gpio_found = False
for name in ip_names:
    if 'gpio' in name.lower():
        gpio_ip = getattr(ol, name)
        print(f"Found GPIO: {name} — using it to retrigger BRAM reader")
        gpio_found = True

        # Assert reset (active high in your Verilog)
        gpio_ip.channel1.write(0x1, 0x1)
        time.sleep(0.05)

        # Re-arm DMA after reset assert
        try:
            dma.recvchannel.transfer(recv_buf)
        except Exception:
            pass

        # Release reset → BRAM reader begins streaming
        gpio_ip.channel1.write(0x0, 0x1)
        print("✓ Reset toggled — BRAM reader now streaming.")
        break

if not gpio_found:
    print("No GPIO found. BRAM reader relies on bitstream-load reset.")
    print("DMA was armed before stream — should be OK.")

# ──────────────────────────────────────────────
# 9. Wait for DMA to complete
# ──────────────────────────────────────────────
timeout = 10
start_t = time.time()
print("\nWaiting for DMA...")

while not dma.recvchannel.idle:
    elapsed = time.time() - start_t
    if elapsed > timeout:
        sr = dma_mmio.read(S2MM_STAT)
        print(f"\nDMA S2MM Status: 0x{sr:08x}")
        print(f"  Halted : {(sr >> 0) & 1}")
        print(f"  Idle   : {(sr >> 1) & 1}")
        print(f"  Error  : {(sr >> 4) & 1}")
        print(f"  Int    : {(sr >> 12) & 1}")
        raise RuntimeError(
            "DMA stuck! BRAM reader is not sending tvalid.\n"
            "Possible causes:\n"
            "  1. rst not connected to GPIO — add AXI GPIO in Vivado\n"
            "  2. BRAM reader already streamed before DMA was armed\n"
            "  3. m_axis_tready not reaching BRAM reader\n"
            "  4. Wrong DMA base address in Address Editor"
        )
    time.sleep(0.01)

print("✓ DMA transfer complete!")

# ──────────────────────────────────────────────
# 10. Verify
# ──────────────────────────────────────────────
received = np.frombuffer(recv_buf, dtype=np.uint8)

print("\n" + "="*52)
print("         DATA TRANSFER SUMMARY")
print("="*52)

mismatch_count = 0
for addr_idx in range(COUNT):
    s    = addr_idx * WORD_BYTES
    raw  = received[s : s + WORD_BYTES]
    rx   = np.frombuffer(raw.tobytes(), dtype=np.uint16)
    ok   = np.array_equal(input_data[addr_idx], rx)
    tag  = "✓ OK  " if ok else "✗ FAIL"
    if not ok:
        mismatch_count += 1
    print(f"  Word[{addr_idx:02d}] {tag} | TX: {input_data[addr_idx]} | RX: {rx}")

print("="*52)
if mismatch_count == 0:
    print(f"  ✓ SUCCESS — All {COUNT} words transferred correctly!")
    print(f"  ✓ {TOTAL_BYTES} bytes: PS→BRAM→BRAMReader→DMA→PS")
else:
    print(f"  ✗ {mismatch_count}/{COUNT} words mismatched.")
print("="*52)

# ──────────────────────────────────────────────
# 11. Cleanup
# ──────────────────────────────────────────────
recv_buf.freebuffer()
print("\nDone.")

Overlay loaded.

Writing data to BRAM...
✓ BRAM write complete.
✓ DMA buffer allocated: 288 bytes
✓ DMA S2MM channel started.

Arming DMA transfer...
Checking for GPIO reset control...
Available IPs: ['axi_dma_0', 'processing_system7_0']
No GPIO found. BRAM reader relies on bitstream-load reset.
DMA was armed before stream — should be OK.

Waiting for DMA...

DMA S2MM Status: 0x00000000
  Halted : 0
  Idle   : 0
  Error  : 0
  Int    : 0


RuntimeError: DMA stuck! BRAM reader is not sending tvalid.
Possible causes:
  1. rst not connected to GPIO — add AXI GPIO in Vivado
  2. BRAM reader already streamed before DMA was armed
  3. m_axis_tready not reaching BRAM reader
  4. Wrong DMA base address in Address Editor